# Target trial emulation: zoster vaccination and all-cause mortality (All of Us)
Sequential trials via risk-set matching: each vaccinee (time zero = first recorded
zoster vaccine dose) is matched on sex at birth and birth year to people who, at that date,
are enrolled, alive, not yet vaccinated and have >= 1 year of prior EHR.
Cohort building runs in BigQuery; weights and Cox models run in pandas/lifelines.
Check every line marked `# CHECK`.

In [ ]:
# 1. Setup and parameters
import os
import numpy as np, pandas as pd
from google.cloud import bigquery

BQ_dataset = os.environ['WORKSPACE_CDR']   # CDR of this workspace (release used: see Methods)
client = bigquery.Client(default_query_job_config=bigquery.QueryJobConfig(default_dataset=BQ_dataset))
def q(sql): return client.query(sql).result().to_dataframe()

CUTOFF      = '2023-10-01'   # data cutoff of CDR R2024Q3R9, used for the manuscript
STUDY_START = '2017-01-01'
AGE_MIN, AGE_MAX = 50, 90
LOOKBACK    = 730            # days for visit/drug covariates; >= 1 visit in this window required
EHR_HISTORY = 365            # days of EHR required before time zero (to see earlier zoster doses)
N_CONTROLS, OVERSAMPLE, SEED = 5, 3, 2026
GRACE_DAYS  = None           # sensitivity: e.g. 365 = censor at last EHR contact + 365 d
RISK_YEARS  = 3
B_BOOT      = 50

lc = "LOWER(concept_name)"
VACC_DOMAIN = f"(domain_id = 'Drug' OR (domain_id = 'Procedure' AND {lc} LIKE '%vaccin%'))"
ZOS = (f"{lc} LIKE '%zoster%' AND {lc} NOT LIKE '%immune globulin%' "
       f"AND {lc} NOT LIKE '%immunoglobulin%' AND {VACC_DOMAIN}")
FLU = f"{lc} LIKE '%influenza%' AND {VACC_DOMAIN}"
TRAVEL = ("(" + " OR ".join(f"{lc} LIKE '%{k}%'" for k in
          ['typhoid', 'hepatitis a', 'hep a', 'yellow fever', 'japanese encephalitis',
           'atovaquone', 'mefloquine']) + f") AND {VACC_DOMAIN}")

In [ ]:
# 2. Check zoster concepts and data cutoff  (run once, inspect, adjust ZOS if needed)
check = q(f"""
WITH zc AS (SELECT concept_id, concept_name, vocabulary_id, domain_id FROM `concept` WHERE {ZOS}),
ev AS (
  SELECT drug_concept_id AS cid, person_id FROM `drug_exposure` WHERE drug_concept_id IN (SELECT concept_id FROM zc)
  UNION ALL SELECT drug_source_concept_id, person_id FROM `drug_exposure` WHERE drug_source_concept_id IN (SELECT concept_id FROM zc)
  UNION ALL SELECT procedure_concept_id, person_id FROM `procedure_occurrence` WHERE procedure_concept_id IN (SELECT concept_id FROM zc)
  UNION ALL SELECT procedure_source_concept_id, person_id FROM `procedure_occurrence` WHERE procedure_source_concept_id IN (SELECT concept_id FROM zc))
SELECT zc.concept_id, zc.concept_name, zc.vocabulary_id, zc.domain_id, COUNT(DISTINCT ev.person_id) AS n_persons
FROM ev JOIN zc ON ev.cid = zc.concept_id
GROUP BY 1, 2, 3, 4 ORDER BY n_persons DESC
""")
display(check)
display(q("""SELECT (SELECT MAX(visit_start_date) FROM `visit_occurrence` WHERE visit_start_date <= CURRENT_DATE()) AS max_visit,
                    (SELECT MAX(death_date) FROM `aou_death`) AS max_death"""))

In [ ]:
# 3. Build the matched cohort with covariates at time zero (one BigQuery script)
CCI = {  # Quan ICD-10 Charlson: name -> (weight, regex on ICD-10-CM code without dot)
  'mi': (1, r'^(I21|I22|I252)'),
  'chf': (1, r'^(I099|I110|I130|I132|I255|I420|I42[5-9]|I43|I50|P290)'),
  'pvd': (1, r'^(I70|I71|I731|I738|I739|I771|I790|I792|K551|K558|K559|Z958|Z959)'),
  'cevd': (1, r'^(G45|G46|H340|I6)'),
  'dementia': (1, r'^(F0[0-3]|F051|G30|G311)'),
  'copd': (1, r'^(I278|I279|J4[0-7]|J6[0-7]|J684|J701|J703)'),
  'rheum': (1, r'^(M05|M06|M315|M3[2-4]|M351|M353|M360)'),
  'pud': (1, r'^K2[5-8]'),
  'liver_mild': (1, r'^(B18|K70[0-39]|K71[3-57]|K73|K74|K760|K76[2-489])'),
  'dm': (1, r'^E1[0-4][01689]'),
  'dm_comp': (2, r'^E1[0-4][23457]'),
  'hemi': (2, r'^(G041|G114|G80[12]|G81|G82|G83[0-49])'),
  'renal': (2, r'^(I120|I131|N03[2-7]|N05[2-7]|N18|N19|N250|Z49[0-2]|Z940|Z992)'),
  'cancer': (2, r'^(C[01][0-9]|C2[0-6]|C3[0-4]|C3[7-9]|C4[01]|C43|C4[5-9]|C5[0-8]|C6[0-9]|C7[0-6]|C8[1-5]|C88|C9[0-7])'),
  'liver_sev': (3, r'^(I850|I859|I864|I982|K704|K711|K721|K729|K76[5-7])'),
  'mets': (6, r'^(C7[7-9]|C80)'),
  'aids': (6, r'^B2[0-24]'),
}
cci_case  = "\n        ".join(f"WHEN REGEXP_CONTAINS(REPLACE(sc.concept_code, '.', ''), r'{rx}') THEN '{nm}'"
                           for nm, (_, rx) in CCI.items())
cci_pivot = ", ".join(f"MAX(IF(x.cat = '{nm}', 1, 0)) AS {nm}" for nm in CCI)
cci_sel   = ", ".join(f"IFNULL(cc.{nm}, 0) AS {nm}" for nm in CCI)

sql = f"""
-- (a) first date of zoster / flu / travel products per person
CREATE TEMP TABLE ev1 AS
WITH cs AS (
  SELECT concept_id,
    CASE WHEN {ZOS} THEN 'zos' WHEN {FLU} THEN 'flu' ELSE 'travel' END AS label,
    CASE WHEN {lc} LIKE '%live%' THEN 'live'
         WHEN {lc} LIKE '%recombinant%' OR {lc} LIKE '%glycoprotein%' OR {lc} LIKE '%adjuvant%' THEN 'recombinant'
         ELSE 'unspecified' END AS vtype
  FROM `concept` WHERE ({ZOS}) OR ({FLU}) OR ({TRAVEL})),
cs_ev AS (
  SELECT de.person_id, de.drug_exposure_start_date AS d, cs.label, cs.vtype FROM `drug_exposure` de JOIN cs ON de.drug_concept_id = cs.concept_id
  UNION ALL SELECT de.person_id, de.drug_exposure_start_date, cs.label, cs.vtype FROM `drug_exposure` de JOIN cs ON de.drug_source_concept_id = cs.concept_id
  UNION ALL SELECT po.person_id, po.procedure_date, cs.label, cs.vtype FROM `procedure_occurrence` po JOIN cs ON po.procedure_concept_id = cs.concept_id
  UNION ALL SELECT po.person_id, po.procedure_date, cs.label, cs.vtype FROM `procedure_occurrence` po JOIN cs ON po.procedure_source_concept_id = cs.concept_id)
SELECT person_id, label, MIN(d) AS first_d,
  ARRAY_AGG(vtype ORDER BY d, IF(vtype = 'unspecified', 1, 0) LIMIT 1)[OFFSET(0)] AS first_vtype
FROM cs_ev WHERE d IS NOT NULL
GROUP BY person_id, label;

-- (b) person-level table
CREATE TEMP TABLE pers AS
WITH enroll AS (
  SELECT o.person_id, MIN(o.observation_date) AS enrollment_date
  FROM `observation` o
  JOIN `concept_ancestor` ca ON o.observation_concept_id = ca.descendant_concept_id
  JOIN `concept` c ON ca.ancestor_concept_id = c.concept_id
  WHERE c.concept_name = 'Consent PII' AND c.concept_class_id = 'Module'
  GROUP BY 1),
dth AS (SELECT person_id, MIN(death_date) AS death_date FROM `aou_death` GROUP BY 1),
vs AS (SELECT person_id, MIN(visit_start_date) AS first_visit, MAX(visit_start_date) AS last_visit
       FROM `visit_occurrence` WHERE visit_start_date <= DATE '{CUTOFF}' GROUP BY 1)
SELECT p.person_id, DATE(p.birth_datetime) AS birth, EXTRACT(YEAR FROM p.birth_datetime) AS byear,
  sx.concept_name AS sex, e.enrollment_date, d.death_date,
  z.first_d AS vacc_date, z.first_vtype AS vacc_type, vs.first_visit, vs.last_visit
FROM `person` p
JOIN `concept` sx ON p.sex_at_birth_concept_id = sx.concept_id      -- CHECK (or gender_concept_id)
JOIN enroll e ON e.person_id = p.person_id
JOIN vs ON vs.person_id = p.person_id
LEFT JOIN dth d ON d.person_id = p.person_id
LEFT JOIN ev1 z ON z.person_id = p.person_id AND z.label = 'zos'
WHERE sx.concept_name IN ('Male', 'Female');

-- (c) vaccinees + risk-set sampled controls (oversampled, final eligibility applied in Python)
CREATE TEMP TABLE cohort0 AS
WITH tr AS (
  SELECT * FROM pers
  WHERE vacc_date >= DATE '{STUDY_START}' AND vacc_date < DATE '{CUTOFF}'
    AND enrollment_date <= vacc_date
    AND (death_date IS NULL OR death_date > vacc_date)
    AND first_visit <= DATE_SUB(vacc_date, INTERVAL {EHR_HISTORY} DAY)
    AND DATE_DIFF(vacc_date, birth, DAY) / 365.25 >= {AGE_MIN}
    AND DATE_DIFF(vacc_date, birth, DAY) / 365.25 <  {AGE_MAX}),
cand AS (
  SELECT t.person_id AS set_id, t.vacc_date AS index_date, c.person_id,
    ROW_NUMBER() OVER (PARTITION BY t.person_id ORDER BY
      FARM_FINGERPRINT(CONCAT(CAST(t.person_id AS STRING), '-', CAST(c.person_id AS STRING), '-{SEED}'))) AS draw
  FROM tr t
  JOIN pers c ON c.sex = t.sex AND c.byear = t.byear AND c.person_id != t.person_id
  WHERE c.enrollment_date <= t.vacc_date
    AND (c.death_date IS NULL OR c.death_date > t.vacc_date)
    AND (c.vacc_date  IS NULL OR c.vacc_date  > t.vacc_date)
    AND c.first_visit <= DATE_SUB(t.vacc_date, INTERVAL {EHR_HISTORY} DAY)
    AND c.last_visit  >= DATE_SUB(t.vacc_date, INTERVAL {LOOKBACK} DAY))
SELECT person_id AS set_id, vacc_date AS index_date, person_id, 1 AS treated, 0 AS draw FROM tr
UNION ALL
SELECT set_id, index_date, person_id, 0 AS treated, draw FROM cand WHERE draw <= {N_CONTROLS * OVERSAMPLE};

-- (d) covariates at time zero
WITH
nvis AS (
  SELECT c.set_id, c.person_id, COUNT(DISTINCT v.visit_start_date) AS n_visits
  FROM cohort0 c JOIN `visit_occurrence` v ON v.person_id = c.person_id
   AND v.visit_start_date >= DATE_SUB(c.index_date, INTERVAL {LOOKBACK} DAY) AND v.visit_start_date < c.index_date
  GROUP BY 1, 2),
ndrug AS (
  SELECT c.set_id, c.person_id, COUNT(DISTINCT de.drug_concept_id) AS n_drugs
  FROM cohort0 c JOIN `drug_exposure` de ON de.person_id = c.person_id
   AND de.drug_exposure_start_date >= DATE_SUB(c.index_date, INTERVAL {LOOKBACK} DAY) AND de.drug_exposure_start_date < c.index_date
  WHERE de.drug_concept_id != 0
  GROUP BY 1, 2),
mark AS (
  SELECT c.set_id, c.person_id, MAX(IF(e.label = 'flu', 1, 0)) AS flu, MAX(IF(e.label = 'travel', 1, 0)) AS travel
  FROM cohort0 c JOIN ev1 e ON e.person_id = c.person_id AND e.first_d < c.index_date AND e.label IN ('flu', 'travel')
  GROUP BY 1, 2),
dx AS (
  SELECT person_id, cat, MIN(d) AS first_d FROM (
    SELECT co.person_id, co.condition_start_date AS d,
      CASE
        {cci_case}
      END AS cat
    FROM `condition_occurrence` co JOIN `concept` sc ON co.condition_source_concept_id = sc.concept_id
    WHERE sc.vocabulary_id = 'ICD10CM')
  WHERE cat IS NOT NULL GROUP BY 1, 2),
cci AS (
  SELECT c.set_id, c.person_id, {cci_pivot}
  FROM cohort0 c JOIN dx x ON x.person_id = c.person_id AND x.first_d < c.index_date
  GROUP BY 1, 2),
svy AS (   -- CHECK question concept ids
  SELECT o.person_id,
    MAX(IF(o.observation_source_concept_id = 1585711, a.concept_name, NULL)) AS srh,       -- general health
    MAX(IF(o.observation_source_concept_id = 1585857, a.concept_name, NULL)) AS smoke100,  -- 100 cigs lifetime
    MAX(IF(o.observation_source_concept_id = 1585940, a.concept_name, NULL)) AS educ       -- highest grade
  FROM `observation` o LEFT JOIN `concept` a ON a.concept_id = o.value_source_concept_id
  WHERE o.observation_source_concept_id IN (1585711, 1585857, 1585940)
  GROUP BY 1),
bmi AS (
  SELECT person_id, ANY_VALUE(value_as_number) AS bmi FROM `measurement`
  WHERE measurement_concept_id = 3038553 AND value_as_number BETWEEN 12 AND 80 GROUP BY 1)
SELECT c.set_id, c.index_date, c.person_id, c.treated, c.draw,
  p.birth, p.sex, p.death_date, p.vacc_date, p.vacc_type, p.last_visit,
  IFNULL(nv.n_visits, 0) AS n_visits, IFNULL(nd.n_drugs, 0) AS n_drugs,
  IFNULL(m.flu, 0) AS flu, IFNULL(m.travel, 0) AS travel,
  {cci_sel}, s.srh, s.smoke100, s.educ, b.bmi
FROM cohort0 c
JOIN pers p ON p.person_id = c.person_id
LEFT JOIN nvis  nv ON nv.set_id = c.set_id AND nv.person_id = c.person_id
LEFT JOIN ndrug nd ON nd.set_id = c.set_id AND nd.person_id = c.person_id
LEFT JOIN mark  m  ON m.set_id  = c.set_id AND m.person_id  = c.person_id
LEFT JOIN cci   cc ON cc.set_id = c.set_id AND cc.person_id = c.person_id
LEFT JOIN svy   s  ON s.person_id = c.person_id
LEFT JOIN bmi   b  ON b.person_id = c.person_id
"""
raw = q(sql)
raw.to_pickle('vzv_aou_cohort_raw.pkl')   # reload later with pd.read_pickle
print(raw.shape, raw.treated.sum())

In [ ]:
# 4. Final eligibility, outcomes, covariates
df = raw.copy()
for c in ['index_date', 'birth', 'death_date', 'vacc_date', 'last_visit']:
    df[c] = pd.to_datetime(df[c])
CUT = pd.Timestamp(CUTOFF)

# >= 1 visit in the prior LOOKBACK days (time-zero eligibility, for everyone)
df = df[df.n_visits >= 1].sort_values(['set_id', 'treated', 'draw'], ascending=[True, False, True])
df['crank'] = (df.treated == 0).astype(int).groupby(df.set_id).cumsum()
df = df[(df.treated == 1) | (df.crank <= N_CONTROLS)]
g = df.groupby('set_id').treated
df = df[(g.transform('max') == 1) & (g.transform('size') > 1)].copy()

# follow-up
end = pd.Series(CUT, index=df.index)
if GRACE_DAYS is not None:
    end = pd.concat([end, df.last_visit + pd.Timedelta(days=GRACE_DAYS)], axis=1).min(axis=1)
df['death'] = (df.death_date.notna() & (df.death_date <= end)).astype(int)
df['end'] = df.death_date.where(df.death == 1, end)
df['time'] = (df.end - df.index_date).dt.days
df = df[df.time > 0].copy()
# per-protocol: controls censored at their own vaccination
pp_cens = (df.treated == 0) & df.vacc_date.notna() & (df.vacc_date < df.end)
df['pp_time'] = np.where(pp_cens, (df.vacc_date - df.index_date).dt.days, df.time)
df['pp_death'] = np.where(pp_cens, 0, df.death)

# Charlson with hierarchy
df.loc[df.dm_comp == 1, 'dm'] = 0
df.loc[df.mets == 1, 'cancer'] = 0
df.loc[df.liver_sev == 1, 'liver_mild'] = 0
df['cci'] = sum(w * df[nm] for nm, (w, _) in CCI.items())
df['cci_cat'] = np.minimum(df.cci, 3).astype(str)
df['cvd_any'] = df[['mi', 'chf', 'pvd', 'cevd']].max(axis=1)
df['dm_any'] = df[['dm', 'dm_comp']].max(axis=1)
df['cancer_any'] = df[['cancer', 'mets']].max(axis=1)
df['age'] = (df.index_date - df.birth).dt.days / 365.25
df['cal_year'] = df.index_date.dt.year.astype(str)
df['lvis'] = np.log1p(df.n_visits)
df['ldrug'] = np.log1p(df.n_drugs)

def srh_map(x):
    x = str(x).lower()
    for k in ['excellent', 'very good', 'good', 'fair', 'poor']:
        if k in x: return k.replace(' ', '_')
    return 'missing'
def smoke_map(x):
    x = str(x).lower().strip()
    return 'ever' if x.endswith('yes') else 'never' if x.endswith('no') else 'missing'
def educ_map(x):
    x = str(x).lower()
    if 'advanced degree' in x or 'college graduate' in x: return 'college_plus'
    if 'college one to three' in x: return 'some_college'
    if 'twelve' in x or 'ged' in x: return 'high_school'
    if any(k in x for k in ['never attended', 'one through', 'five through', 'nine through']): return 'lt_high_school'
    return 'missing'
df['srh_c'] = df.srh.map(srh_map)
df['smoke_c'] = df.smoke100.map(smoke_map)
df['educ_c'] = df.educ.map(educ_map)
df['bmi_c'] = pd.cut(df.bmi, [0, 25, 30, 200], labels=['lt25', '25_30', 'ge30']).astype(str).replace('nan', 'missing')

print('Sets:', df.set_id.nunique(), ' Vaccinees:', df.treated.sum(), ' Controls:', (df.treated == 0).sum())
print(df.groupby('treated').agg(n=('death', 'size'), deaths=('death', 'sum'),
      py=('time', lambda t: t.sum() / 365.25), age=('age', 'mean'), cci=('cci', 'mean'),
      visits=('n_visits', 'mean')).round(2))
print(pd.crosstab(df.srh_c, df.treated, normalize='columns').round(3))

In [ ]:
# 5. Propensity scores and ATT weights (without / with self-rated health)
import statsmodels.api as sm, statsmodels.formula.api as smf
PS_BASE = ('treated ~ bs(age, df=3) + C(cal_year) + C(cci_cat) + lvis + ldrug + flu + travel'
           ' + cvd_any + dm_any + cancer_any + copd + dementia + renal'
           ' + C(smoke_c) + C(educ_c) + C(bmi_c)')
PS_SRH = PS_BASE + ' + C(srh_c)'

def att_weights(formula):
    ps = smf.glm(formula, data=df, family=sm.families.Binomial()).fit().predict(df)
    w = np.where(df.treated == 1, 1.0, ps / (1 - ps))
    cap = np.quantile(w[df.treated == 0], 0.99)
    return np.where((df.treated == 0) & (w > cap), cap, w)

df['w0'] = att_weights(PS_BASE)
df['w'] = att_weights(PS_SRH)          # primary

def smd(x, t, w):
    m1, m0 = np.average(x[t == 1], weights=w[t == 1]), np.average(x[t == 0], weights=w[t == 0])
    return (m1 - m0) / np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2)
bal_df = df.assign(**{f'srh_{k}': (df.srh_c == k).astype(int) for k in ['excellent', 'very_good', 'good', 'fair', 'poor']},
                   ever_smoke=(df.smoke_c == 'ever').astype(int))
bal_vars = ['age', 'n_visits', 'n_drugs', 'cci', 'flu', 'travel', 'cvd_any', 'dm_any', 'cancer_any',
            'copd', 'dementia', 'renal', 'ever_smoke', 'srh_excellent', 'srh_very_good', 'srh_good', 'srh_fair', 'srh_poor']
t, one = bal_df.treated.values, np.ones(len(bal_df))
balance = pd.DataFrame({v: {'smd_unweighted': smd(bal_df[v].values, t, one),
                            'smd_w_no_srh': smd(bal_df[v].values, t, bal_df.w0.values),
                            'smd_w_srh': smd(bal_df[v].values, t, bal_df.w.values)} for v in bal_vars}).T.round(3)
display(balance)

In [ ]:
# 6. Hazard ratios
from lifelines import CoxPHFitter
def cox(d, time='time', event='death', w='w', covars=(), label=''):
    cph = CoxPHFitter()
    cph.fit(d[[time, event, 'treated', w, 'person_id', *covars]], duration_col=time, event_col=event,
            weights_col=w, cluster_col='person_id', robust=True)
    s = cph.summary.loc['treated']
    return {'analysis': label, 'HR': s['exp(coef)'], 'lo': s['exp(coef) lower 95%'],
            'hi': s['exp(coef) upper 95%'], 'events_vacc': int(d.loc[d.treated == 1, event].sum()),
            'events_ctrl': int(d.loc[d.treated == 0, event].sum())}

def window(d, a, b):
    x = d[d.time > a].copy()
    x['wtime'] = np.minimum(x.time, b) - a
    x['wev'] = ((x.death == 1) & (x.time <= b)).astype(int)
    return x

res = [
    cox(df.assign(one=1.0), w='one', label='Unweighted (matched only)'),
    cox(df, w='w0', label='ITT, PS without self-rated health'),
    cox(df, label='ITT, PS with self-rated health (primary)'),
    cox(df, covars=('age', 'cci', 'lvis', 'ldrug', 'flu', 'travel'), label='ITT, weighted + adjusted'),
    cox(df, time='pp_time', event='pp_death', label='Per-protocol (controls censored at vaccination)'),
]
for a, b, lab in [(0, 90, '0-90 d'), (90, 365, '91-365 d'), (365, np.inf, '>1 y')]:
    res.append(cox(window(df, a, b), time='wtime', event='wev', label=f'ITT window {lab}'))
df['set_type'] = df.set_id.map(df[df.treated == 1].set_index('set_id').vacc_type)
for vt in ['recombinant', 'live']:
    d = df[df.set_type == vt]
    if d.loc[d.treated == 1, 'death'].sum() >= 20:
        res.append(cox(d, label=f'ITT, {vt} vaccine'))
results = pd.DataFrame(res).round(3)
display(results)

In [ ]:
# 7. Absolute risk at RISK_YEARS (weighted KM), bootstrap by matched set
from lifelines import KaplanMeierFitter
T = RISK_YEARS * 365.25
def risks(d):
    r = {}
    for tr in (0, 1):
        x = d[d.treated == tr]
        km = KaplanMeierFitter().fit(x.time, x.death, weights=x.w)
        r[tr] = 1 - float(km.survival_function_at_times(T).iloc[0])
    return np.array([r[0], r[1], r[1] - r[0]])
pt = risks(df)
rng = np.random.default_rng(SEED)
sets = df.set_id.unique()
by_set = {k: v for k, v in df.groupby('set_id')}
boot = np.array([risks(pd.concat([by_set[s] for s in rng.choice(sets, len(sets))])) for _ in range(B_BOOT)])
risk_tab = pd.DataFrame({'estimate': pt, 'lo': np.quantile(boot, 0.025, axis=0), 'hi': np.quantile(boot, 0.975, axis=0)},
                        index=[f'risk_ctrl_{RISK_YEARS}y', f'risk_vacc_{RISK_YEARS}y', 'risk_difference'])
display(risk_tab.round(4))

In [ ]:
# 8. E-value (deaths are rare in AoU, so RR ~ HR)
def evalue(rr):
    rr = 1 / rr if rr < 1 else rr
    return rr + np.sqrt(rr * (rr - 1))
prim = results[results.analysis.str.contains('primary')].iloc[0]
print('E-value point:', round(evalue(prim.HR), 2),
      ' CI limit:', 1.0 if prim.lo <= 1 <= prim.hi else round(evalue(prim.hi if prim.HR < 1 else prim.lo), 2))

In [ ]:
# 9. Cumulative mortality plot and export
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4.5))
for tr, col, lab in [(0, 'grey', 'Not vaccinated'), (1, '#2a7f62', 'Zoster vaccine')]:
    x = df[df.treated == tr]
    km = KaplanMeierFitter().fit(x.time / 365.25, x.death, weights=x.w)
    ax.step(km.timeline, 1 - km.survival_function_.iloc[:, 0], where='post', color=col, lw=2, label=lab)
ax.set_xlabel('Years since time zero'); ax.set_ylabel('Cumulative mortality'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig('vzv_aou_cumulative_mortality.png', dpi=200)
results.to_csv('vzv_aou_hazard_ratios.csv', index=False)
balance.to_csv('vzv_aou_balance.csv')
risk_tab.to_csv('vzv_aou_risk.csv')
# AoU dissemination policy: do not publish any count between 1 and 20.